In [10]:
import os
import csv
import glob
import logging
import tempfile
import subprocess
from pathlib import Path
from typing import Tuple

import pdfplumber
from pypdf import PdfReader

## Logging Setup

In [11]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
log = logging.getLogger(__name__)

### Config path

In [12]:
DATASET_DIR      = Path("..\Dataset")       # root containing occupation subfolders
OUTPUT_DIR       = Path("..\Extracted_data")     # mirrored output tree (.txt files)
LOG_FILE         = Path("..\extraction_log.csv")
MIN_CHARS_NATIVE = 50                    # char threshold to classify as native PDF
OCR_DPI          = 200                   # DPI for rasterising scanned pages

### Detection

In [13]:
def is_native_pdf(pdf_path: Path, sample_pages: int = 2) -> bool:
    """
    Returns True if the PDF has a readable text layer (native/text-based).
    Samples the first `sample_pages` pages; if extracted text exceeds
    MIN_CHARS_NATIVE characters, it is treated as native.
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = "".join(
                (page.extract_text() or "") for page in pdf.pages[:sample_pages]
            )
        return len(text.strip()) >= MIN_CHARS_NATIVE
    except Exception:
        return False  # if we can't read it at all, fall back to OCR


### PDF Extraction

In [14]:
def extract_native(pdf_path: Path) -> Tuple[str, int]:
    """
    Direct text-layer extraction using pdfplumber.
    Returns (full_text, page_count).
    """
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        page_count = len(pdf.pages)
        for i, page in enumerate(pdf.pages):
            t = page.extract_text() or ""
            if t.strip():
                pages_text.append(f"--- Page {i + 1} ---\n{t}")
    return "\n\n".join(pages_text), page_count

### Master extraction

In [15]:
def extract_pdf(pdf_path: Path) -> dict:
    """
    Detects PDF type and routes to the appropriate extractor.
    Returns a result dict including the extracted text and metadata.
    """
    result = {
        "occupation": pdf_path.parent.name,
        "filename":   pdf_path.name,
        "file":       str(pdf_path),
        "method":     None,
        "page_count": 0,
        "char_count": 0,
        "status":     "ok",
        "error":      "",
        "text":       "",
    }

    try:
        native = is_native_pdf(pdf_path)
        if native:
            text, pages = extract_native(pdf_path)
            result["method"] = "native_pdfplumber"
        else:
            log.info(f"-> Scanned PDF detected, applying OCR: {pdf_path.name}")
            text, pages = extract_scanned(pdf_path)
            result["method"] = "ocr_pytesseract"

        result["text"]       = text
        result["page_count"] = pages
        result["char_count"] = len(text)

        if not text.strip():
            result["status"] = "empty"
            result["error"]  = "No text could be extracted"

    except Exception as exc:
        result["status"] = "error"
        result["error"]  = str(exc)
        log.error(f"Error processing {pdf_path.name}: {exc}")

    return result

### Extract files

In [17]:
def run(dataset_dir: Path = DATASET_DIR,
        output_dir:  Path = OUTPUT_DIR,
        log_file:    Path = LOG_FILE):

    if not dataset_dir.exists():
        log.error(f"Dataset directory not found: {dataset_dir.resolve()}")
        return

    occupation_dirs = [d for d in sorted(dataset_dir.iterdir()) if d.is_dir()]
    if not occupation_dirs:
        log.warning("No occupation subdirectories found.")
        return

    log.info(f"Dataset: {dataset_dir.resolve()}")
    log.info(f"Occupations found: {len(occupation_dirs)}")

    log_rows = []

    for occ_dir in occupation_dirs:
        pdf_files = sorted(occ_dir.glob("*.pdf"))
        if not pdf_files:
            log.info(f"[skip] No PDFs in: {occ_dir.name}")
            continue

        out_occ = output_dir / occ_dir.name
        out_occ.mkdir(parents=True, exist_ok=True)

        log.info(f"\n[{occ_dir.name}] {len(pdf_files)} PDF(s)")

        for pdf_path in pdf_files:
            log.info(f"Processing: {pdf_path.name}")
            result = extract_pdf(pdf_path)

            # Save extracted text
            out_txt = out_occ / (pdf_path.stem + ".txt")
            out_txt.write_text(result["text"], encoding="utf-8")

            # Accumulate log row (strip raw text)
            log_rows.append({k: v for k, v in result.items() if k != "text"})

            icon = "v" if result["status"] == "ok" else "x"
            log.info(
                f"{icon} [{result['method']}] "
                f"{result['page_count']} page(s) | "
                f"{result['char_count']:,} chars -> {out_txt.name}"
            )

    # Write extraction log
    if log_rows:
        with open(log_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=log_rows[0].keys())
            writer.writeheader()
            writer.writerows(log_rows)
        log.info(f"\nExtraction log: {log_file}")

    ok    = sum(1 for r in log_rows if r["status"] == "ok")
    total = len(log_rows)
    errs  = total - ok
    log.info(
        f"Done — {ok}/{total} files extracted successfully" + (f" | {errs} error(s) — check extraction_log.csv" if errs else ""))


if __name__ == "__main__":
    run()

2026-05-19 12:52:26,812 [INFO] Dataset: C:\Binus\CV-Screening-RM\Dataset
2026-05-19 12:52:26,813 [INFO] Occupations found: 24
2026-05-19 12:52:26,814 [INFO] 
[ACCOUNTANT] 118 PDF(s)
2026-05-19 12:52:26,815 [INFO] Processing: 10554236.pdf
2026-05-19 12:52:31,040 [INFO] v [native_pdfplumber] 5 page(s) | 24,232 chars -> 10554236.txt
2026-05-19 12:52:31,040 [INFO] Processing: 10674770.pdf
2026-05-19 12:52:32,671 [INFO] v [native_pdfplumber] 2 page(s) | 7,519 chars -> 10674770.txt
2026-05-19 12:52:32,672 [INFO] Processing: 11163645.pdf
2026-05-19 12:52:33,901 [INFO] v [native_pdfplumber] 2 page(s) | 4,773 chars -> 11163645.txt
2026-05-19 12:52:33,902 [INFO] Processing: 11759079.pdf
2026-05-19 12:52:35,478 [INFO] v [native_pdfplumber] 2 page(s) | 5,948 chars -> 11759079.txt
2026-05-19 12:52:35,478 [INFO] Processing: 12065211.pdf
2026-05-19 12:52:36,932 [INFO] v [native_pdfplumber] 2 page(s) | 5,592 chars -> 12065211.txt
2026-05-19 12:52:36,933 [INFO] Processing: 12202337.pdf
2026-05-19 12:52